# LangChain: Q&A over Documents

An example might be a tool that would allow you to query a product catalog for items of interest.

本节演示"检索增强生成"（RAG）的最基础形态：把文档切分、向量化后存入向量数据库，
再用相似度检索找出跟问题相关的片段，最后交给 LLM 结合这些片段回答问题。

> 注：本 notebook 依赖的 `OutdoorClothingCatalog_1000.csv` 数据文件在当前目录下不存在（课程平台提供），
> 涉及读取该文件、生成 embedding、向量检索的 cell 无法在本地完整跑通——这是数据/网络缺失问题，不是代码逻辑 bug。
> 这里仍然修复了所有版本兼容性问题，并在能本地验证的部分（如 import、构造函数签名）做了实际验证。

In [ ]:
# 加载 .env 中的环境变量（如 OPENAI_API_KEY）
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [ ]:
# account for deprecation of LLM model
# 根据当前日期判断该用哪个 gpt-3.5-turbo 版本名，逻辑本身没问题
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
# 【版本兼容性修复】原代码全部从 langchain.xxx 导入，这些子模块在当前 langchain 1.4.0 中已不存在：
#   RetrievalQA        -> langchain_classic.chains（经典链）
#   ChatOpenAI, OpenAI  -> langchain_openai（独立子包）
#   CSVLoader           -> langchain_community.document_loaders
#   DocArrayInMemorySearch -> langchain_community.vectorstores
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown
from langchain_openai import OpenAI

In [ ]:
# 加载产品目录 CSV，CSVLoader 会把每一行转换成一个 LangChain Document 对象
# 【环境限制】OutdoorClothingCatalog_1000.csv 在当前目录下不存在，这里无法实际运行，不是代码 bug
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)

In [ ]:
# 【版本兼容性修复】原代码 from langchain.indexes import VectorstoreIndexCreator 已不存在，
# 正确路径是 langchain_classic.indexes
from langchain_classic.indexes import VectorstoreIndexCreator

In [ ]:
# VectorstoreIndexCreator 是一个"一站式"封装：自动完成 加载文档 -> 切分 -> 向量化 -> 存入向量库 的流程
# 【真实 bug 修复】原代码没有传入 embedding 参数。旧版 langchain 的 VectorstoreIndexCreator
# 会默认使用 OpenAIEmbeddings() 作为 embedding；但在当前 langchain_classic 版本中，
# embedding 字段变成了必填项（没有默认值），不传会直接报
# "pydantic.ValidationError: 1 validation error ... embedding Field required"。
# 这里显式传入 OpenAIEmbeddings() 修复该问题（已在本地用 venv 验证：不传会报错，传了才能正常构造）。
from langchain_openai import OpenAIEmbeddings

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=OpenAIEmbeddings(),
).from_loaders([loader])

In [ ]:
# 要问的问题：列出所有带防晒功能的衬衫，并总结每一款
query ="Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

In [ ]:
# 用一个纯文本补全模型（不是 Chat 模型）作为替代，index.query() 内部会自动帮你完成"检索 + 生成"整个流程
llm_replacement_model = OpenAI(temperature=0,
                               model='gpt-3.5-turbo-instruct')

response = index.query(query,
                       llm = llm_replacement_model)

In [ ]:
# 以 Markdown 格式渲染模型返回的表格
display(Markdown(response))

In [ ]:
# 接下来手动走一遍 VectorstoreIndexCreator 内部做的事情，帮助理解 RAG 的每一步
# 【版本兼容性修复】同前，CSVLoader 正确路径是 langchain_community.document_loaders
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader(file_path=file)

In [ ]:
# .load() 真正读取文件，返回 Document 对象列表（每行 CSV 对应一个 Document）
docs = loader.load()

In [ ]:
# 查看第一条 Document，page_content 是这一行 CSV 转换成的文本，metadata 里会带上来源信息
docs[0]

In [ ]:
# 【版本兼容性修复】原代码 from langchain.embeddings import OpenAIEmbeddings 已不存在，
# 正确路径是 langchain_openai
# OpenAIEmbeddings：把文本转换成一个高维向量，语义相近的文本向量距离也相近
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# 对一句话做 embedding，得到一个向量（这一步需要真实调用 OpenAI 的 embedding 接口，联网）
embed = embeddings.embed_query("Hi my name is Harrison")

In [ ]:
# 查看向量维度（text-embedding-ada-002 通常是 1536 维）
print(len(embed))

In [ ]:
# 看看向量的前 5 个数值长什么样（浮点数）
print(embed[:5])

In [ ]:
# from_documents 会对每个 Document 调用 embeddings 生成向量，并存入内存向量库 DocArrayInMemorySearch
db = DocArrayInMemorySearch.from_documents(
    docs,
    embeddings
)

In [ ]:
query = "Please suggest a shirt with sunblocking"

In [ ]:
# similarity_search：把 query 也转成向量，在向量库里找出最相似的几条 Document（默认返回 4 条）
docs = db.similarity_search(query)

In [ ]:
len(docs)

In [ ]:
docs[0]

In [ ]:
# as_retriever() 把向量库包装成一个标准的 Retriever 接口，可以直接插进 RetrievalQA 等 Chain 里使用
retriever = db.as_retriever()

In [ ]:
llm = ChatOpenAI(temperature = 0.0, model=llm_model)

In [ ]:
# 把检索到的几条 Document 的正文拼接成一整段文本，作为"上下文"塞进 prompt 里
qdocs = "".join([docs[i].page_content for i in range(len(docs))])


In [ ]:
# 【真实 bug 修复】原代码调用 llm.call_as_llm(...)。
# call_as_llm 是旧版 ChatOpenAI 上的一个便捷方法，用来"把 chat 模型当成普通文本补全模型"直接传字符串调用。
# 当前 langchain_openai.ChatOpenAI 已经没有这个方法了（已在本地 venv 验证：hasattr(llm, 'call_as_llm') 为 False），
# 调用会直接抛 AttributeError。正确写法是用 .invoke() 传入一条消息（或消息列表），
# 再从返回的 AIMessage 里取 .content。
response = llm.invoke(f"{qdocs} Question: Please list all your \
shirts with sun protection in a table in markdown and summarize each one.").content


In [ ]:
display(Markdown(response))

In [ ]:
# RetrievalQA.from_chain_type 把"检索 + 拼接上下文 + 调用 LLM"这一整套流程封装成一个 Chain，
# chain_type="stuff" 表示直接把所有检索到的文档"塞"进一个 prompt 里（最简单的策略，适合文档较少的场景）
qa_stuff = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True
)

In [ ]:
query =  "Please list all your shirts with sun protection in a table \
in markdown and summarize each one."

In [ ]:
# 【提示】.run() 是 Chain 的旧式调用方法，当前版本仍可用但已 deprecated，新写法是 .invoke({"query": query})
response = qa_stuff.run(query)

In [ ]:
display(Markdown(response))

In [ ]:
# 用之前的 index.query 一步到位地问同样的问题，效果应该和 qa_stuff 手动搭建的流程一致
response = index.query(query, llm=llm)

In [ ]:
# 这里再次演示"显式传入 embedding"的写法（呼应前面 e2849112c310596f 那个 cell 的修复）：
# 在当前 langchain_classic 版本中，VectorstoreIndexCreator 的 embedding 参数是必填的
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])